# Week 3: Modeling II, Networks, Transportation, Transshipment and Process Superstructures

**Course:** 2105623 Optimization of Chemical Processes  
**Institution:** Department of Chemical Engineering, Chulalongkorn University  
**Instructor:** Assoc. Prof. Dr. Soorathep Kheawhom

**Week:** 3 of 15 (modeling block, Weeks 1 to 6)  
**CLO mapping:** CLO 1 (formulation), CLO 4 (implementation and solution in Pyomo)

## Learning objectives

By the end of this notebook you should be able to:

- Write the node balance equation that every network flow model shares, and recognize the node-arc incidence matrix behind it.
- Formulate the transportation, transshipment, shortest path and maximum flow problems as special cases of one minimum cost flow template.
- Explain why network LPs return integer optimal solutions without any integer variable, using total unimodularity, and say what destroys that property.
- Read node potentials and arc reduced costs from the LP duals, identify a minimum cut from a maximum flow solution, and cross-check an LP network solution against a combinatorial algorithm implemented from scratch.
- Represent competing process routes as a **superstructure**, that is a network of units and streams with continuous splits and material balances, and state exactly what turns that linear superstructure into an MINLP.

**Estimated duration:** 110 minutes  
**Prerequisites:** Week 1 (the five-stage workflow, model classification) and Week 2 (index sets, balance equations, capacity constraints, linking constraints), elementary graph vocabulary, LP duality at an introductory level.

**Reference:** Rao, Ch. 3; Williams, *Model Building in Mathematical Programming*, Ch. 5; Ahuja, Magnanti and Orlin, *Network Flows*, Ch. 1 to 6; Biegler, Grossmann and Westerberg, *Systematic Methods of Chemical Process Design*, Ch. 9 (superstructure optimization).


In [ ]:
# --- Environment check -------------------------------------------------------
import sys, subprocess, importlib, shutil

def ensure(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg])

for p, n in [("pyomo", "pyomo"), ("numpy", "numpy"), ("scipy", "scipy"),
             ("matplotlib", "matplotlib"), ("pandas", "pandas")]:
    ensure(p, n)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import pyomo.environ as pyo

def pick_solver(kind="lp"):
    """Return the first available solver of the requested kind."""
    order = {"lp":   ["appsi_highs", "glpk", "cbc", "gurobi", "cplex"],
             "milp": ["appsi_highs", "cbc", "glpk", "gurobi", "cplex"],
             "nlp":  ["ipopt", "conopt", "knitro"],
             "minlp":["bonmin", "couenne", "mindtpy"]}[kind]
    for name in order:
        try:
            s = pyo.SolverFactory(name)
            if s is not None and s.available(exception_flag=False):
                print(f"Using solver: {name}")
                return s
        except Exception:
            continue
    raise RuntimeError(f"No {kind} solver found. Install one, e.g. 'pip install highspy' "
                       f"or 'conda install -c conda-forge ipopt glpk coincbc'.")

In [ ]:
# --- Figure style: Teal-Amber Lab Palette v1.0 -------------------------------
PALETTE = ["#0F6E6B", "#E29A2D", "#BE654C", "#5A91BE", "#83A462", "#995A90", "#333F4A", "#DFC98F"]
INK, GRAPHITE, MIST, PAPER = "#1C242B", "#333F4A", "#B9C1C6", "#F3F0EB"

plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 150,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": GRAPHITE, "axes.labelcolor": INK, "axes.titlecolor": INK,
    "axes.linewidth": 1.0, "axes.grid": True, "axes.axisbelow": True,
    "grid.color": MIST, "grid.linewidth": 0.7, "grid.alpha": 0.9,
    "xtick.color": GRAPHITE, "ytick.color": GRAPHITE,
    "text.color": INK, "lines.linewidth": 1.8, "lines.markersize": 5,
    "font.size": 9, "legend.frameon": False,
    "axes.prop_cycle": plt.cycler(color=PALETTE),
})

def tidy(ax):
    """Apply the house style to a single Axes object."""
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRAPHITE)
    return ax

print("Palette loaded:", ", ".join(PALETTE[:3]), "...")

## 1. The one template

Every model in this notebook is an instance of the **minimum cost flow** problem. Let `G = (N, A)` be a
directed graph with nodes `N` and arcs `A`. For each arc `(i,j)` let `f_ij >= 0` be the flow, `c_ij` the cost
per unit of flow, and `u_ij` the arc capacity. For each node `n` let `b_n` be its net supply, positive at a
source, negative at a sink, zero at a pure transit node.

    min   sum_{(i,j) in A} c_ij f_ij
    s.t.  sum_{j : (n,j) in A} f_nj  -  sum_{i : (i,n) in A} f_in  =  b_n     for all n in N
          0 <= f_ij <= u_ij                                                    for all (i,j) in A

The equality is the **node balance**: outflow minus inflow equals net supply. Written in matrix form the
constraint is `E f = b`, where `E` is the **node-arc incidence matrix**, with `E[i, (i,j)] = +1`,
`E[j, (i,j)] = -1` and zero elsewhere. Every column of `E` has exactly one `+1` and one `-1`.

The four classical problems are obtained by choosing the graph and the data:

| Problem | Graph | Costs | Capacities | Supplies |
|---|---|---|---|---|
| Transportation | bipartite, sources to sinks | unit shipping cost | usually none | supply and demand |
| Transshipment | any, with transit nodes | unit shipping cost | arc limits | supply and demand |
| Shortest path | any | arc length | none | `+1` at origin, `-1` at destination |
| Maximum flow | any, plus super source and super sink | zero, except `-1` on a return arc | arc limits | zero everywhere |

A feasibility condition applies to the whole class: `sum_n b_n = 0`. Summing the node balances gives zero on
the left, because every arc contributes `+f_ij` at its tail and `-f_ij` at its head, so total supply must
equal total demand. When it does not, a dummy node absorbs the difference.

## 2. Application: a battery-materials distribution network

Three precursor plants (`R1`, `R2`, `R3`) supply cathode active material to four cell gigafactories
(`G1` to `G4`). Two coastal consolidation terminals (`H1`, `H2`) can receive material and re-ship it.
Quantities are tonnes per week, costs are USD per tonne.

In [ ]:
# --- Transportation data: plants direct to gigafactories ---------------------
S = ["R1", "R2", "R3"]
D = ["G1", "G2", "G3", "G4"]

supply = pd.Series({"R1": 450, "R2": 380, "R3": 300}, name="supply_t")
demand = pd.Series({"G1": 300, "G2": 260, "G3": 330, "G4": 240}, name="demand_t")

cost_tab = pd.DataFrame(
    [[18, 24, 31, 40],
     [27, 19, 22, 33],
     [38, 30, 25, 21]], index=S, columns=D)

print("unit shipping cost, USD/t\n", cost_tab, "\n")
print(supply.to_frame().T)
print(demand.to_frame().T)
print(f"\ntotal supply {supply.sum()} t, total demand {demand.sum()} t -> "
      f"{'balanced' if supply.sum() == demand.sum() else 'unbalanced'}")

### 2.1 Transportation model

    min   sum_i sum_j c_ij x_ij
    s.t.  sum_j x_ij <= s_i           for all plants i        (supply)
          sum_i x_ij >= d_j           for all gigafactories j (demand)
          x_ij >= 0

This is the bipartite special case of the template. Writing supply with `<=` and demand with `>=` keeps the
model feasible even when total supply exceeds total demand. Here the instance happens to be balanced, so both
families hold with equality at the optimum.

In [ ]:
# --- Transportation model ----------------------------------------------------
def build_transport():
    m = pyo.ConcreteModel(name="transportation")
    m.S = pyo.Set(initialize=S)
    m.D = pyo.Set(initialize=D)
    m.c = pyo.Param(m.S, m.D, initialize={(i, j): float(cost_tab.loc[i, j]) for i in S for j in D})
    m.s = pyo.Param(m.S, initialize=supply.to_dict())
    m.d = pyo.Param(m.D, initialize=demand.to_dict())
    m.x = pyo.Var(m.S, m.D, domain=pyo.NonNegativeReals)
    m.supply = pyo.Constraint(m.S, rule=lambda m, i: sum(m.x[i, j] for j in m.D) <= m.s[i])
    m.demand = pyo.Constraint(m.D, rule=lambda m, j: sum(m.x[i, j] for i in m.S) >= m.d[j])
    m.cost = pyo.Objective(expr=sum(m.c[i, j]*m.x[i, j] for i in m.S for j in m.D),
                           sense=pyo.minimize)
    m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    return m

lp = pick_solver("lp")
mt = build_transport()
rt = lp.solve(mt)
print(rt.solver.termination_condition)
assert rt.solver.termination_condition == pyo.TerminationCondition.optimal, \
    "transportation LP did not solve to optimality"
print(f"minimum shipping cost: {pyo.value(mt.cost):,.2f} USD/week")

---

### The same model in the spreadsheet

The companion workbook for this week is `excel/W03_transportation_STUDENT.xlsx`. Per the tool allocation
table of the course specification, Week 3 is an **OpenSolver week for the matrix models and a Python week
for the graph models**: a cost matrix is a spreadsheet, and shortest path and maximum flow are not. The
`Transportation` sheet holds the bipartite problem as a matrix; the arc-list model lives on a second sheet
named `Transshipment`, and the change of layout between the two sheets is itself the lesson.

| Algebraic symbol | Spreadsheet range or layout | Pyomo component | Note |
|---|---|---|---|
| `i in S`, plants | row labels `B12:B14` on sheet `Transportation` | `m.S = pyo.Set(initialize=S)` | Sources index the rows of every block on the sheet. |
| `j in D`, gigafactories | column headers `C11:F11` | `m.D = pyo.Set(initialize=D)` | Sinks index the columns. The two-dimensional index is the reason this model belongs in a sheet. |
| `c_ij`, unit shipping cost | cost matrix `C6:F8`, blue text, row labels `B6:B8` and headers `C5:F5` | `m.c = pyo.Param(m.S, m.D, initialize=...)` | The one object in the whole notebook that a spreadsheet displays better than code. |
| `s_i`, supply | `Supply` column `I12:I14`, blue text | `m.s = pyo.Param(m.S, initialize=supply.to_dict())` | |
| `d_j`, demand | `Demand` row `C17:F17`, blue text | `m.d = pyo.Param(m.D, initialize=demand.to_dict())` | |
| `x_ij >= 0`, tonnes shipped | changing cells `C12:F14`, green fill, the same 3 by 4 shape as the cost matrix | `m.x = pyo.Var(m.S, m.D, domain=pyo.NonNegativeReals)` | Shape matching is what makes the objective a single `SUMPRODUCT` over two rectangles. |
| `sum_j x_ij <= s_i` | `VALUE` column `G12:G14`, `=SUM(C12:F12)`; relation column `H12:H14` holding `<=`; supply column `I12:I14` | `m.supply = pyo.Constraint(m.S, rule=...)` | Dialog entry `$G$12:$G$14 <= $I$12:$I$14`. |
| `sum_i x_ij >= d_j` | `VALUE` row `C15:F15`, `=SUM(C12:C14)`; relation row `C16:F16` holding `>=`; demand row `C17:F17` | `m.demand = pyo.Constraint(m.D, rule=...)` | One family runs down the sheet and the other across it. In Pyomo both are the same kind of object with a different index set. |
| `min sum_i sum_j c_ij x_ij` | objective cell `C19`, `=SUMPRODUCT(C6:F8,C12:F14)`, entered with To: Min | `m.cost = pyo.Objective(expr=..., sense=pyo.minimize)` | A `SUMPRODUCT` of two rectangles is the sheet's only two-dimensional operator. |
| `u_i`, `v_j`, the MODI potentials | Sensitivity report, Constraints table, `Shadow Price` column | `m.dual[m.supply[i]]`, `m.dual[m.demand[j]]` | The reduced cost `c_ij - u_i - v_j` has to be built as a further block of formulas on the sheet; the STUDENT workbook does not contain one. |
| `(i,j) in A`, arcs of the transshipment network | the matrix is abandoned: on sheet `Transshipment`, `C6:C23` tail, `D6:D23` head, `E6:E23` cost, `F6:F23` capacity, `G6:G23` flow in green | `m.A = pyo.Set(initialize=arc_list, dimen=2)` | 9 nodes and 18 arcs would be a 9 by 18 incidence block of which 126 of the 162 entries are structurally zero. |
| `E f = b`, node balance | one row per node in `B29:B37`: outflow `C29 = SUMIF($C$6:$C$23,B29,$G$6:$G$23)` minus inflow `D29 = SUMIF($D$6:$D$23,B29,$G$6:$G$23)`, differenced in `E29 = C29-D29`; relation column `F29:F37`; net supply `G29:G37` | `m.balance = pyo.Constraint(m.N, rule=balance_rule)` | The incidence matrix is never written down. A pair of `SUMIF`s with a hand-typed criterion stands in for each row of `E`. |
| `min sum c_ij f_ij` | objective cell `C25`, `=SUMPRODUCT(E6:E23,G6:G23)` | `m.cost = pyo.Objective(...)` | Same operator, one-dimensional ranges this time. |
| `0 <= f_ij <= u_ij` | dialog entry `$G$6:$G$23 <= $F$6:$F$23` | `bounds=lambda m, i, j: (0, ARCS[i, j][1])` on `m.f` | In Pyomo the capacity travels with the variable declaration and cannot be forgotten in the dialog. |
| `b_n` for shortest path and maximum flow | the same net supply column `G29:G37`, retyped once per query | the `b` argument of `build_flow` | One function argument in Python; one manual edit and one re-solve per origin-destination pair in the sheet. |

**Where the spreadsheet stops working.** The transportation model is the case where the spreadsheet is
genuinely the better tool: the cost data already is a 3 by 4 matrix, the changing cells are a matrix of
the same shape, and the two constraint families are its row sums and its column sums. The transshipment
network breaks that picture, because a network is not a matrix. Its 9 nodes and 18 arcs would need a
9 by 18 incidence block that is 126 zeros out of 162 entries, so the model has to be rebuilt as an arc
list and each node balance becomes a hand-written pair of `SUMIF`s whose criterion string must be typed
correctly nine times. Shortest path is worse. The LP is unchanged, but the object wanted is a **path**:
the sheet returns a column of 18 flows, and the route from `R1` to `G4` has to be reassembled by eye
from whichever entries are nonzero, because no cell anywhere in the workbook holds a path. Asking for a
second destination means retyping the `b` column and solving again, one destination per solve, where
Section 3 changes one dictionary. Maximum flow is worse again: it needs a super source, a super sink and
one new arc per plant and per gigafactory, that is seven new rows in a list that must also keep the
halved terminal capacities straight, and the minimum cut is a **partition of the nodes**, an object with
no representation in a cell at all. Section 5 closes the argument: the total unimodularity spot check is
20,000 submatrix determinants, and no Solver dialog can compute one of them.

---

In [ ]:
# --- Transportation results --------------------------------------------------
ship = pd.DataFrame([[pyo.value(mt.x[i, j]) for j in D] for i in S], index=S, columns=D)
ship["shipped"] = ship.sum(axis=1)
ship.loc["received"] = list(ship[D].sum()) + [ship["shipped"].sum()]
print(ship.round(4), "\n")

u = pd.Series({i: mt.dual[mt.supply[i]] for i in S}, name="u_i (supply dual)")
v = pd.Series({j: mt.dual[mt.demand[j]] for j in D}, name="v_j (demand dual)")
print(u.round(4).to_frame().T)
print(v.round(4).to_frame().T)

# complementary slackness: reduced cost c_ij - u_i - v_j is zero on every used arc
rc = pd.DataFrame([[float(cost_tab.loc[i, j]) - u[i] - v[j] for j in D] for i in S],
                  index=S, columns=D).round(6)
print("\nreduced cost c_ij - u_i - v_j\n", rc)
used = [(i, j) for i in S for j in D if pyo.value(mt.x[i, j]) > 1e-9]
assert all(abs(rc.loc[i, j]) < 1e-6 for i, j in used), \
    "a shipped arc has non-zero reduced cost"
print("\nevery arc carrying flow has reduced cost zero, as complementary slackness requires")
print("all shipments integral:",
      bool(np.allclose([pyo.value(mt.x[i, j]) for i in S for j in D],
                       np.round([pyo.value(mt.x[i, j]) for i in S for j in D]))))

### 2.2 The transshipment network

Direct plant-to-gigafactory shipping is expensive for the long lanes. The company also operates two coastal
terminals. Material can move plant to terminal, terminal to terminal, and terminal to gigafactory, and two
direct lanes remain available. Every arc now has a capacity, and the terminals are pure transit nodes with
`b_n = 0`.

The model is the general template written verbatim: one balance per node, one bound per arc.

In [ ]:
# --- Network data: arcs, unit cost, capacity ---------------------------------
NODES = ["R1", "R2", "R3", "H1", "H2", "G1", "G2", "G3", "G4"]

ARCS = {
    ("R1", "H1"): (12, 400), ("R1", "H2"): (20, 250),
    ("R2", "H1"): (16, 300), ("R2", "H2"): (14, 350),
    ("R3", "H1"): (24, 200), ("R3", "H2"): (11, 300),
    ("H1", "H2"): ( 5, 200), ("H2", "H1"): ( 5, 200),
    ("H1", "G1"): ( 8, 350), ("H1", "G2"): (11, 300),
    ("H1", "G3"): (17, 250), ("H1", "G4"): (26, 200),
    ("H2", "G1"): (22, 250), ("H2", "G2"): (13, 300),
    ("H2", "G3"): ( 9, 350), ("H2", "G4"): (12, 300),
    ("R1", "G1"): (30, 150), ("R3", "G4"): (28, 120),
}

b_net = {"R1": 450, "R2": 380, "R3": 300, "H1": 0, "H2": 0,
         "G1": -300, "G2": -260, "G3": -330, "G4": -240}

arc_tab = pd.DataFrame([{"arc": f"{i}->{j}", "cost": c, "capacity": u}
                        for (i, j), (c, u) in ARCS.items()])
print(arc_tab.to_string(index=False))
print(f"\nnodes {len(NODES)}, arcs {len(ARCS)}, sum of net supplies {sum(b_net.values())}")

In [ ]:
# --- Node-arc incidence matrix E ---------------------------------------------
arc_list = list(ARCS.keys())
E = np.zeros((len(NODES), len(arc_list)))
for k, (i, j) in enumerate(arc_list):
    E[NODES.index(i), k] = +1.0
    E[NODES.index(j), k] = -1.0

E_df = pd.DataFrame(E.astype(int), index=NODES,
                    columns=[f"{i}{j}" for i, j in arc_list])
print("node-arc incidence matrix (rows = nodes, columns = arcs)")
print(E_df.to_string())
print("\nevery column has exactly one +1 and one -1:",
      bool(np.all(E.sum(axis=0) == 0) and np.all(np.abs(E).sum(axis=0) == 2)))

In [ ]:
# --- Minimum cost transshipment ----------------------------------------------
def build_flow(b, capacitated=True, cost_key=0, name="min_cost_flow"):
    m = pyo.ConcreteModel(name=name)
    m.N = pyo.Set(initialize=NODES)
    m.A = pyo.Set(initialize=arc_list, dimen=2)
    if capacitated:
        m.f = pyo.Var(m.A, domain=pyo.NonNegativeReals,
                      bounds=lambda m, i, j: (0, ARCS[i, j][1]))
    else:
        m.f = pyo.Var(m.A, domain=pyo.NonNegativeReals)

    def balance_rule(m, n):
        out = sum(m.f[i, j] for (i, j) in m.A if i == n)
        inn = sum(m.f[i, j] for (i, j) in m.A if j == n)
        return out - inn == b.get(n, 0)
    m.balance = pyo.Constraint(m.N, rule=balance_rule)

    m.cost = pyo.Objective(expr=sum(ARCS[a][cost_key]*m.f[a] for a in m.A),
                           sense=pyo.minimize)
    m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    return m

mf = build_flow(b_net)
rf = lp.solve(mf)
print(rf.solver.termination_condition)
assert rf.solver.termination_condition == pyo.TerminationCondition.optimal, \
    "transshipment LP did not solve to optimality"
flow_opt = {a: pyo.value(mf.f[a]) for a in arc_list}
print(f"minimum transshipment cost: {pyo.value(mf.cost):,.2f} USD/week")

res_arcs = pd.DataFrame([{"arc": f"{i}->{j}", "flow": round(flow_opt[i, j], 4),
                          "capacity": ARCS[i, j][1], "cost": ARCS[i, j][0],
                          "saturated": flow_opt[i, j] > ARCS[i, j][1] - 1e-6}
                         for (i, j) in arc_list if flow_opt[i, j] > 1e-9])
print(res_arcs.to_string(index=False))

In [ ]:
# --- Node potentials and arc reduced costs -----------------------------------
pi = pd.Series({n: mf.dual[mf.balance[n]] for n in NODES}, name="potential")
print("node potentials (LP duals of the balance constraints)")
print(pi.round(4).to_frame().T, "\n")

rows = []
for (i, j) in arc_list:
    rows.append({"arc": f"{i}->{j}", "cost": ARCS[i, j][0],
                 "reduced_cost": round(ARCS[i, j][0] - (pi[i] - pi[j]), 6),
                 "flow": round(flow_opt[i, j], 3),
                 "capacity": ARCS[i, j][1]})
red = pd.DataFrame(rows)
print(red.to_string(index=False))
print("\nOptimality conditions for min cost flow:")
print("  reduced cost > 0  -> arc must be empty")
print("  reduced cost < 0  -> arc must be saturated")
print("  reduced cost = 0  -> arc may carry any flow between its bounds")
viol = [r for _, r in red.iterrows()
        if (r.reduced_cost > 1e-6 and r.flow > 1e-6)
        or (r.reduced_cost < -1e-6 and r.flow < r.capacity - 1e-6)]
assert not viol, f"complementary slackness violated on {viol}"
print("checked: no violation")

In [ ]:
# --- Figure 1: the distribution network with optimal flows -------------------
POS = {"R1": (0.0, 2.75), "R2": (0.0, 1.55), "R3": (0.0, 0.35),
       "H1": (1.7, 2.30), "H2": (1.7, 0.80),
       "G1": (3.4, 3.20), "G2": (3.4, 2.25), "G3": (3.4, 1.30), "G4": (3.4, 0.35)}

def draw_network(ax, flows, title, hi_arcs=(), show_labels=True):
    fmax = max(max(flows.values()), 1.0)
    for (i, j) in arc_list:
        f = flows[(i, j)]
        x0, y0 = POS[i]; x1, y1 = POS[j]
        if f > 1e-9:
            col = PALETTE[2] if (i, j) in hi_arcs else (PALETTE[1] if i.startswith("R") and j.startswith("G") else PALETTE[0])
            lw, al, zo = 1.2 + 4.2*f/fmax, 0.95, 3
        else:
            col, lw, al, zo = MIST, 1.0, 0.8, 1
        rad = 0.16 if (i, j) == ("H2", "H1") else (-0.16 if (i, j) == ("H1", "H2") else 0.0)
        ax.annotate("", xy=(x1, y1), xytext=(x0, y0), zorder=zo,
                    arrowprops=dict(arrowstyle="-|>", color=col, lw=lw, alpha=al,
                                    shrinkA=13, shrinkB=13,
                                    connectionstyle=f"arc3,rad={rad}"))
        if show_labels and f > 1e-9:
            # stagger the label along each arc so that crossing lanes do not collide
            s_pos = 0.30 if i.startswith("R") else (0.70 if j.startswith("G") else 0.50)
            ax.text(x0 + s_pos*(x1-x0) + 0.5*rad, y0 + s_pos*(y1-y0) + 0.09 + 0.35*rad,
                    f"{f:.0f}", fontsize=6.4, color=INK, ha="center",
                    bbox=dict(fc="white", ec="none", pad=0.6, alpha=0.9), zorder=4)
    for n, (x, y) in POS.items():
        fc = PALETTE[0] if n.startswith("R") else (PALETTE[6] if n.startswith("H") else PALETTE[1])
        ax.add_patch(plt.Circle((x, y), 0.16, fc=fc, ec=GRAPHITE, lw=1.0, zorder=5))
        ax.text(x, y, n, color="white", fontsize=7.5, ha="center", va="center",
                zorder=6, fontweight="bold")
    ax.set_xlim(-0.45, 3.95); ax.set_ylim(-0.10, 3.75)
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(title, fontsize=10, color=INK)

fig, ax = plt.subplots(figsize=(7.4, 4.6), constrained_layout=True)
draw_network(ax, flow_opt,
             f"Week 3, Figure 1: minimum cost transshipment, "
             f"{pyo.value(mf.cost):,.0f} USD/week")
ax.text(0.0, 3.62, "plants", fontsize=8, color=GRAPHITE, ha="center")
ax.text(1.7, 3.62, "terminals", fontsize=8, color=GRAPHITE, ha="center")
ax.text(3.4, 3.62, "gigafactories", fontsize=8, color=GRAPHITE, ha="center")
ax.text(-0.4, -0.08, "teal = via terminal, amber = direct lane, grey = unused arc",
        fontsize=7.5, color=GRAPHITE, ha="left")
plt.show()

## 3. Shortest path as a minimum cost flow

Send exactly one unit from an origin to a destination and let the arc costs be lengths. The node balance
becomes `b_origin = +1`, `b_destination = -1`, and `b_n = 0` elsewhere. Capacities are dropped, because one
unit can never exceed a sensible capacity.

Nothing else changes. The optimal flow is a single path, because the incidence matrix forces an integral
basic optimal solution and the only integral unit flows are paths. This is the first payoff of total
unimodularity: an inherently combinatorial object comes out of a linear program.

Here the arc costs are re-interpreted as transit times in hours, and we look for the fastest route from plant
`R1` to gigafactory `G4`.

In [ ]:
# --- Shortest path R1 -> G4 by LP -------------------------------------------
b_sp = {n: 0 for n in NODES}
b_sp["R1"], b_sp["G4"] = 1, -1

msp = build_flow(b_sp, capacitated=False, name="shortest_path")
rsp = lp.solve(msp)
assert rsp.solver.termination_condition == pyo.TerminationCondition.optimal, \
    "shortest path LP did not solve to optimality"

path_arcs = [a for a in arc_list if pyo.value(msp.f[a]) > 1e-9]
lp_len = pyo.value(msp.cost)
print(f"LP optimal length : {lp_len:.4f} h")
print("arcs carrying flow:", [f"{i}->{j} ({pyo.value(msp.f[i,j]):.3f})" for i, j in path_arcs])

In [ ]:
# --- Cross-check: Dijkstra implemented from scratch --------------------------
def dijkstra(source, target, arcs):
    """Textbook Dijkstra on a directed graph given as {(i,j): (cost, cap)}."""
    length = {n: np.inf for n in NODES}
    prev = {n: None for n in NODES}
    length[source] = 0.0
    unvisited = set(NODES)
    while unvisited:
        n = min(unvisited, key=lambda k: length[k])
        if length[n] == np.inf:
            break
        unvisited.discard(n)
        for (i, j), (c, _) in arcs.items():
            if i == n and j in unvisited and length[n] + c < length[j]:
                length[j] = length[n] + c
                prev[j] = n
    route, n = [], target
    while n is not None:
        route.append(n)
        n = prev[n]
    return length[target], list(reversed(route))

dj_len, dj_route = dijkstra("R1", "G4", ARCS)
print(f"Dijkstra length: {dj_len:.4f} h  route: {' -> '.join(dj_route)}")
print(f"LP length      : {lp_len:.4f} h")
assert abs(dj_len - lp_len) < 1e-9, "LP and Dijkstra disagree on the shortest path length"

alt = {"R1 -> H1 -> G4": ARCS[("R1","H1")][0] + ARCS[("H1","G4")][0],
       "R1 -> H2 -> G4": ARCS[("R1","H2")][0] + ARCS[("H2","G4")][0],
       "R1 -> H1 -> H2 -> G4": ARCS[("R1","H1")][0] + ARCS[("H1","H2")][0] + ARCS[("H2","G4")][0],
       "R1 -> G1 (dead end for G4)": np.inf}
print("\nall simple R1 to G4 routes:")
for k, v in alt.items():
    print(f"   {k:<28s} {v}")

## 4. Maximum flow and the minimum cut

Now ask a capacity question rather than a cost question. Both terminals are undergoing berth maintenance,
which halves the capacity of every terminal-to-gigafactory arc. How much material can the network still
deliver in a week?

Add a super source `SRC` with an arc to each plant of capacity equal to its supply, and a super sink `SNK`
with an arc from each gigafactory of capacity equal to its demand. Maximize the total flow into `SNK`.

    max   sum_j t_j
    s.t.  inflow(n) + s_n = outflow(n) + t_n     for all n in N
          0 <= f_ij <= u_ij,  0 <= s_i <= supply_i,  0 <= t_j <= demand_j

The **max-flow min-cut theorem** states that the maximum flow equals the minimum capacity of a cut, that is,
of a partition of the nodes into a set containing `SRC` and a set containing `SNK`, where the cut capacity is
the total capacity of the arcs crossing from the first set to the second. This is LP duality specialized to
a network, and the minimum cut can be read off the duals of the capacity constraints.

In [ ]:
# --- Maximum flow under halved terminal outbound capacity --------------------
cap_red = {a: (int(0.5*u) if a[0] in ("H1", "H2") and a[1].startswith("G") else u)
           for a, (c, u) in ARCS.items()}
print("reduced capacities on terminal outbound arcs:")
print({f"{i}->{j}": cap_red[(i, j)] for (i, j) in arc_list
       if i in ("H1", "H2") and j.startswith("G")})

mx = pyo.ConcreteModel(name="max_flow")
mx.N = pyo.Set(initialize=NODES)
mx.A = pyo.Set(initialize=arc_list, dimen=2)
mx.f = pyo.Var(mx.A, domain=pyo.NonNegativeReals)
mx.s = pyo.Var(S, domain=pyo.NonNegativeReals, bounds=lambda m, i: (0, supply[i]))
mx.t = pyo.Var(D, domain=pyo.NonNegativeReals, bounds=lambda m, j: (0, demand[j]))
mx.arc_cap = pyo.Constraint(mx.A, rule=lambda m, i, j: m.f[i, j] <= cap_red[(i, j)])

def bal_mx(m, n):
    out = sum(m.f[i, j] for (i, j) in m.A if i == n)
    inn = sum(m.f[i, j] for (i, j) in m.A if j == n)
    return inn + (m.s[n] if n in S else 0) == out + (m.t[n] if n in D else 0)
mx.balance = pyo.Constraint(mx.N, rule=bal_mx)
mx.throughput = pyo.Objective(expr=sum(mx.t[j] for j in D), sense=pyo.maximize)
mx.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)

rx = lp.solve(mx)
assert rx.solver.termination_condition == pyo.TerminationCondition.optimal, \
    "max flow LP did not solve to optimality"
maxflow = pyo.value(mx.throughput)
print(f"\nmaximum weekly throughput: {maxflow:,.1f} t  (unrestricted demand is {demand.sum()} t)")
print("delivered per gigafactory:", {j: round(pyo.value(mx.t[j]), 2) for j in D})
print("shortfall per gigafactory:", {j: round(demand[j] - pyo.value(mx.t[j]), 2) for j in D})
print("drawn per plant          :", {i: round(pyo.value(mx.s[i]), 2) for i in S})

In [ ]:
# --- Minimum cut: from the duals, and by exhaustive enumeration --------------
cut_from_dual = [f"{i}->{j}" for (i, j) in arc_list if abs(mx.dual[mx.arc_cap[i, j]]) > 1e-7]
print("arcs whose capacity dual is non-zero (the cut inside the network):", cut_from_dual)

import itertools
ext = ([(i, j, cap_red[(i, j)]) for (i, j) in arc_list]
       + [("SRC", i, float(supply[i])) for i in S]
       + [(j, "SNK", float(demand[j])) for j in D])

best_cap, best_set = np.inf, None
for r in range(len(NODES) + 1):
    for sub in itertools.combinations(NODES, r):
        Sset = set(sub) | {"SRC"}
        capsum = sum(w for (i, j, w) in ext if i in Sset and j not in Sset)
        if capsum < best_cap:
            best_cap, best_set = capsum, Sset

print(f"\nminimum cut capacity by enumeration of all {2**len(NODES)} cuts: {best_cap:,.1f} t")
print("source side of the cut:", sorted(best_set))
print("arcs crossing the cut:")
for (i, j, w) in ext:
    if i in best_set and j not in best_set:
        print(f"   {i:>4s} -> {j:<4s} capacity {w:6.0f}")
assert abs(best_cap - maxflow) < 1e-6, "max-flow min-cut theorem violated"
print("\nmax flow equals min cut, as the theorem requires")

In [ ]:
# --- Figure 2: max flow, saturated arcs and the bottleneck -------------------
flow_mx = {a: pyo.value(mx.f[a]) for a in arc_list}
cut_arcs = tuple((i, j) for (i, j) in arc_list if i in best_set and j not in best_set)

fig, axes = plt.subplots(1, 2, figsize=(10.6, 4.3), constrained_layout=True,
                         gridspec_kw={"width_ratios": [1.55, 1.0]})

draw_network(axes[0], flow_mx, f"Maximum flow {maxflow:,.0f} t/week", hi_arcs=cut_arcs)
axes[0].text(-0.4, -0.08, "rust = arcs of the minimum cut", fontsize=7.5,
             color=GRAPHITE, ha="left")

ax = tidy(axes[1])
idx = np.arange(len(D)); w = 0.38
ax.bar(idx - w/2, [demand[j] for j in D], width=w, color=PALETTE[0], label="demand")
ax.bar(idx + w/2, [pyo.value(mx.t[j]) for j in D], width=w, color=PALETTE[1],
       label="delivered at max flow")
for k, j in enumerate(D):
    gap = demand[j] - pyo.value(mx.t[j])
    if gap > 1e-6:
        ax.text(k + w/2, pyo.value(mx.t[j]) + 6, f"-{gap:.0f}", ha="center",
                fontsize=8, color=PALETTE[2])
ax.set_xticks(idx); ax.set_xticklabels(D)
ax.set_ylabel("tonnes per week")
ax.set_title("Where the shortfall lands", fontsize=10)
ax.set_ylim(0, 400)
ax.legend(fontsize=8, loc="upper left")

fig.suptitle("Week 3, Figure 2: maximum flow and the minimum cut", color=INK, fontsize=10)
plt.show()

## 5. Total unimodularity and integral solutions

A matrix `E` is **totally unimodular** (TU) if the determinant of every square submatrix is `0`, `+1` or
`-1`. The node-arc incidence matrix of a directed graph is TU. A short proof is by induction on the size of
the submatrix: a square submatrix either contains a column with no non-zero entry (determinant zero), or a
column with a single non-zero entry (expand along it and apply the induction hypothesis), or every column has
both a `+1` and a `-1`, in which case the rows sum to zero and the determinant is zero.

The consequence is the reason network models are so useful.

**Theorem (Hoffman and Kruskal).** If `E` is TU and the vectors `b` and `u` are integral, then every basic
feasible solution of `{f : E f = b, 0 <= f <= u}` is integral. In particular the simplex method, which always
returns a basic solution, returns an integer answer.

Why: a basic solution solves `B f_B = b'` for a nonsingular square submatrix `B` of `E`, so by Cramer's rule
`f_B = B^{-1} b'` with `det(B)` equal to `+1` or `-1`, and `B^{-1}` has integer entries. An integer matrix times an integer
vector is an integer vector.

Practical significance:

- Transportation, transshipment, shortest path, maximum flow and assignment problems never need integer
  variables. Solve the LP, get integers, at LP speed.
- The result is a property of the **constraint matrix**, not of the costs. Changing the costs cannot make the
  answer fractional.
- TU is fragile. Add one constraint that is not part of the incidence structure, for example a joint budget
  across arcs or a fixed charge with a binary, and the property is lost. The LP relaxation then generally has
  fractional vertices and branch and bound becomes necessary. Week 11 returns to this point.

In [ ]:
# --- Numerical evidence that E is totally unimodular -------------------------
rng = np.random.default_rng(2026)
dets = []
for _ in range(20000):
    k = int(rng.integers(1, min(7, len(NODES)) + 1))
    rows_i = rng.choice(len(NODES), size=k, replace=False)
    cols_i = rng.choice(len(arc_list), size=k, replace=False)
    dets.append(int(round(np.linalg.det(E[np.ix_(rows_i, cols_i)]))))

dets = np.array(dets)
print("distinct determinant values found in 20000 random square submatrices:",
      sorted(set(dets.tolist())))
assert set(dets.tolist()) <= {-1, 0, 1}, "found a submatrix determinant outside {-1,0,1}"
print("counts:", {v: int((dets == v).sum()) for v in sorted(set(dets.tolist()))})
print("\nE passes the total unimodularity spot check.")

In [ ]:
# --- Integrality holds for the LP, and breaks when TU is destroyed -----------
print("A. transshipment LP flows")
vals = np.array([flow_opt[a] for a in arc_list])
print("   max distance from an integer:", float(np.abs(vals - np.round(vals)).max()))
assert np.allclose(vals, np.round(vals), atol=1e-7)

print("\nB. same network, but with a joint constraint that is not part of E")
# Terminal H1 may handle at most 35 percent of everything that leaves the plants.
SHARE = 0.35
m_tu = build_flow(b_net, name="flow_with_side_constraint")
m_tu.side = pyo.Constraint(
    expr=sum(m_tu.f[i, j] for (i, j) in arc_list if j == "H1")
         <= SHARE*sum(m_tu.f[i, j] for (i, j) in arc_list if i in S))
r_tu = lp.solve(m_tu)
assert r_tu.solver.termination_condition == pyo.TerminationCondition.optimal
vals2 = np.array([pyo.value(m_tu.f[a]) for a in arc_list])
frac = np.abs(vals2 - np.round(vals2))
print(f"   optimal cost: {pyo.value(m_tu.cost):,.4f} USD/week")
print(f"   max distance from an integer: {frac.max():.6f}")
print("   fractional arcs:",
      [f"{i}->{j} = {pyo.value(m_tu.f[i,j]):.4f}" for (i, j) in arc_list
       if abs(pyo.value(m_tu.f[i, j]) - round(pyo.value(m_tu.f[i, j]))) > 1e-7])
assert frac.max() > 1e-6, "expected fractional flows once TU is destroyed"
print("\nOne side constraint outside the incidence structure is enough to lose integrality.")

## 6. Interpretation of the network models

**Transportation.** The optimum ships 23,570.00 USD per week. The duals split naturally into a plant value
`u_i` and a gigafactory value `v_j` with `c_ij - u_i - v_j = 0` on every lane that carries flow and `>= 0` on
every lane that does not. That is exactly the classical `u-v` or MODI optimality test, obtained here for free
from the LP dual rather than from a hand-built tableau. Economically, `v_j` is the delivered value of one
extra tonne at gigafactory `j`, and `u_i` the value of one extra tonne of capacity at plant `i`.

**Transshipment.** Routing through the terminals costs 25,850.00 USD per week against 23,570.00 USD for the
direct bipartite problem, but the two numbers are not comparable: the direct problem has unlimited lane
capacities and no terminal handling, while the transshipment network has an arc capacity on every lane. The
transshipment solution saturates `R1 -> H1` at 400 t and `R3 -> H2` at 300 t, which is where the pressure is.
Arc reduced costs `c_ij - (pi_i - pi_j)` obey the standard sign rules everywhere, and the notebook asserts it.

**Shortest path.** The LP returns a length of 29 hours on the route `R1 -> H1 -> H2 -> G4`, matching the
from-scratch Dijkstra implementation exactly. The inter-terminal arc, cheap at 5 hours, beats both direct
terminal routes: `R1 -> H1 -> G4` costs 38 and `R1 -> H2 -> G4` costs 32. Note that the LP produced a
0-1 flow without being told to.

**Maximum flow.** With terminal outbound capacity halved the network delivers 1,100 t per week against
1,130 t of demand. The entire 30 t shortfall lands on `G3`. The minimum cut, found both from the LP duals and
by enumerating all 512 cuts, has capacity 1,100 t and consists of the two arcs into `G3`
(`H1 -> G3` at 125 t and `H2 -> G3` at 175 t) together with the three sink arcs for `G1`, `G2` and `G4`.
The engineering reading is precise: adding capacity anywhere except on an arc of the cut changes nothing.
To recover the missing 30 t you must widen `H1 -> G3` or `H2 -> G3`, and no other investment will do it.

**Integrality.** Every flow in the transportation, transshipment, shortest path and maximum flow solutions
came out integral from a pure LP solve. Section 5 then adds one innocuous-looking side constraint, a 35
percent share limit on terminal `H1`. The cost rises to 26,195.00 USD per week and four arcs come out
at exactly one half tonne, for example `R1 -> H1 = 365.5` and `H1 -> G2 = 95.5`. Total unimodularity is a
property of the constraint matrix and it does not survive arbitrary additions.

## 7. Process superstructures

Sections 2 to 5 treated a network whose nodes are places. A **process superstructure** is the same mathematics with the nodes reinterpreted as process units and the arcs as material streams. It is the standard way to pose a process synthesis question, that is, a question of the form "which flowsheet should we build", rather than "how should we operate the flowsheet we have".

The idea is to build one network that contains **every candidate route at once**, and then let the optimizer decide how much material goes down each route. A route that is not worth using simply receives zero flow. Nothing about that decision requires a binary variable: the split of a stream between competing units is a continuous variable, bounded below by zero, and the optimizer drives the uninteresting splits to zero by itself. The pattern vocabulary of Week 2 covers the whole formulation:

- **index set** over units and over streams,
- **balance equation** around every unit and every mixing or splitting node,
- **capacity constraint** on every unit,
- **linking constraint** wherever two routes share a downstream unit.

### Stage 1: the verbal problem

A battery-materials site needs 100 t/day of purified hydrogen for cathode-precursor reduction. Three production routes are available: steam methane reforming (`SMR`), autothermal reforming (`ATR`) and water electrolysis (`ELY`). `SMR` and `ATR` produce a crude hydrogen stream that must be purified; `ELY` produces hydrogen that is already pure enough and needs no purification. Two purification technologies are available for the crude stream: pressure swing adsorption (`PSA`), which recovers more hydrogen but costs more to run, and a membrane unit (`MEM`), which is cheaper and recovers less. Crude hydrogen from either reformer may go to either purifier in any proportion. Choose the throughput of every unit so that the 100 t/day requirement is met at least daily operating cost.

### Stage 2: the structured specification

| Object | Symbol | Meaning | Units |
|---|---|---|---|
| index set | `u in U` | process unit, `U = {SMR, ATR, ELY, PSA, MEM}` | - |
| parameter | `k_u` | operating cost of unit `u` per tonne processed | USD/t |
| parameter | `Fmax_u` | design capacity of unit `u` | t/day |
| parameter | `eta_u` | hydrogen recovery of purifier `u` at design throughput | t pure / t crude |
| parameter | `D` | purified hydrogen required | t/day |
| decision | `F_u >= 0` | throughput of unit `u` | t/day |

Restrictions, in words:

- R1. Every tonne of crude hydrogen leaving the two reformers enters exactly one of the two purifiers (crude node balance).
- R2. Purified hydrogen from `PSA` and `MEM` plus the direct electrolysis stream must meet the requirement (product node balance).
- R3. No unit may exceed its design capacity.
- R4. All throughputs are non-negative.

### Stage 3: the algebraic model, linear version

    minimize    sum_{u in U} k_u F_u                                    [USD/day]

    subject to  F_SMR + F_ATR = F_PSA + F_MEM                           crude balance,  [t/day]
                eta_PSA F_PSA + eta_MEM F_MEM + F_ELY = D               product balance,[t/day]
                0 <= F_u <= Fmax_u                for all u in U        capacity,       [t/day]

There is no split fraction in this formulation, and there does not need to be one. The split of crude hydrogen between `PSA` and `MEM` is `F_PSA / (F_SMR + F_ATR)`, and it is recovered after the solve rather than imposed before it. Writing splits as ratios would introduce a division and make the model nonlinear for no gain: this is the first and cheapest lesson of superstructure modeling.

The two balances are the linking constraints. The crude balance links the reformer block to the purifier block, and the product balance links all three routes to the single requirement. Delete them and the model separates into five unrelated one-variable problems with no feasible answer.


In [ ]:
# --- Stage 4: superstructure data and the linear model -----------------------
UNITS   = ["SMR", "ATR", "ELY", "PSA", "MEM"]
STAGE_OF = {"SMR": "production", "ATR": "production", "ELY": "production",
            "PSA": "purification", "MEM": "purification"}

CAP     = {"SMR":  90.0, "ATR":  70.0, "ELY":  40.0, "PSA": 100.0, "MEM":  80.0}  # t/day
OPEX    = {"SMR": 1300.0, "ATR": 1550.0, "ELY": 3400.0, "PSA": 200.0, "MEM": 100.0}  # USD/t
ETA_DES = {"PSA": 0.88, "MEM": 0.80}          # recovery at design throughput, t pure / t crude
DEMAND  = 100.0                                # t/day of purified hydrogen

print(pd.DataFrame({"stage": STAGE_OF, "capacity_t_per_day": CAP,
                    "opex_USD_per_t": OPEX,
                    "recovery": {u: ETA_DES.get(u, np.nan) for u in UNITS}}).to_string())
print(f"\nrequirement: {DEMAND:.0f} t/day of purified hydrogen")

# Effective cost per tonne of PURIFIED hydrogen along each route, computed by hand
# so that the LP answer can be predicted before it is produced.
routes = []
for prod in ["SMR", "ATR"]:
    for pur in ["PSA", "MEM"]:
        routes.append({"route": f"{prod} -> {pur}",
                       "USD per t pure": (OPEX[prod] + OPEX[pur]) / ETA_DES[pur]})
routes.append({"route": "ELY (no purification)", "USD per t pure": OPEX["ELY"]})
print("\ncost of one tonne of purified hydrogen, by route")
print(pd.DataFrame(routes).sort_values("USD per t pure").round(2).to_string(index=False))


def build_super(binaries=False, relax=False, nonlinear=False, fix_y=None,
                name="superstructure"):
    """Superstructure model. binaries=True adds unit selection, nonlinear=True makes
    purifier recovery depend on load, which is what turns the MILP into an MINLP.
    FIXED is introduced in Section 7.1 and ETA0, ETA_SLOPE in Section 7.2; they are
    read only when the corresponding option is switched on."""
    m = pyo.ConcreteModel(name=name)
    m.U = pyo.Set(initialize=UNITS)
    m.F = pyo.Var(m.U, domain=pyo.NonNegativeReals,
                  bounds=lambda m, u: (0.0, CAP[u]), initialize=1.0)      # t/day

    if binaries:
        m.y = pyo.Var(m.U, domain=(pyo.UnitInterval if relax else pyo.Binary),
                      initialize=1.0)
        m.select = pyo.Constraint(m.U, rule=lambda m, u: m.F[u] <= CAP[u] * m.y[u])
        if fix_y is not None:
            for u in UNITS:
                m.y[u].fix(float(fix_y[u]))

    # R1 crude balance: everything the reformers make enters a purifier
    m.crude = pyo.Constraint(expr=m.F["SMR"] + m.F["ATR"] == m.F["PSA"] + m.F["MEM"])

    # R2 product balance
    if nonlinear:
        pure = (sum((ETA0[u] - ETA_SLOPE[u] * m.F[u]) * m.F[u] for u in ("PSA", "MEM"))
                + m.F["ELY"])
    else:
        pure = sum(ETA_DES[u] * m.F[u] for u in ("PSA", "MEM")) + m.F["ELY"]
    m.product = pyo.Constraint(expr=pure == DEMAND)

    obj = sum(OPEX[u] * m.F[u] for u in m.U)
    if binaries:
        obj = obj + sum(FIXED[u] * m.y[u] for u in m.U)
    m.cost = pyo.Objective(expr=obj, sense=pyo.minimize)
    if not binaries or relax:
        m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    return m

In [ ]:
# --- Stage 5: solve the linear superstructure and check it -------------------
m_lp = build_super(name="superstructure_LP")
r_lp = lp.solve(m_lp)
print(r_lp.solver.termination_condition)
assert r_lp.solver.termination_condition == pyo.TerminationCondition.optimal, \
    "linear superstructure did not solve to optimality"

F_lp = {u: pyo.value(m_lp.F[u]) for u in UNITS}
cost_lp = float(pyo.value(m_lp.cost))
print(f"minimum operating cost: {cost_lp:,.2f} USD/day "
      f"({cost_lp / DEMAND:,.2f} USD per tonne of purified hydrogen)\n")

sup = pd.DataFrame({"throughput_t_per_day": F_lp, "capacity_t_per_day": CAP,
                    "utilization": {u: F_lp[u] / CAP[u] for u in UNITS},
                    "opex_USD_per_day": {u: OPEX[u] * F_lp[u] for u in UNITS}})
sup["selected"] = sup.throughput_t_per_day > 1e-9
print(sup.round(4).to_string())

# The split fractions are recovered after the solve, not imposed before it.
crude = F_lp["SMR"] + F_lp["ATR"]
print(f"\ncrude hydrogen produced: {crude:.2f} t/day")
print(f"  split to PSA: {F_lp['PSA'] / crude:.4f}   split to MEM: {F_lp['MEM'] / crude:.4f}"
      f"   (sum {(F_lp['PSA'] + F_lp['MEM']) / crude:.4f})")

# Material balance audit, independent of the solver.
bal_crude = F_lp["SMR"] + F_lp["ATR"] - F_lp["PSA"] - F_lp["MEM"]
bal_prod = (ETA_DES["PSA"] * F_lp["PSA"] + ETA_DES["MEM"] * F_lp["MEM"]
            + F_lp["ELY"] - DEMAND)
print(f"\nmaterial balance residuals: crude {bal_crude:+.3e} t/day, "
      f"product {bal_prod:+.3e} t/day")
assert abs(bal_crude) < 1e-6 and abs(bal_prod) < 1e-6, "a unit balance does not close"
assert all(F_lp[u] <= CAP[u] + 1e-6 for u in UNITS), "a unit exceeds its capacity"

# The dual of the product balance is the marginal cost of the last tonne delivered.
marg = m_lp.dual[m_lp.product]
route_cost = {r["route"]: r["USD per t pure"] for r in routes}
print(f"\ndual of the product balance : {marg:,.4f} USD per extra tonne of pure hydrogen")
print(f"cost of the ATR -> MEM route: {route_cost['ATR -> MEM']:,.4f} USD per tonne")
print("The marginal tonne is made by the most expensive route still carrying flow.")
assert abs(marg - route_cost["ATR -> MEM"]) < 1e-6, \
    "the product dual does not match the marginal route cost"

### 7.1 Adding unit selection: the superstructure becomes an MILP

The linear model above answers an operating question: given that all five units exist, how should material be routed. Process synthesis asks a harder question: which units should be built at all. A unit that is built carries a capital charge whether or not it is loaded, and that charge is a fixed cost, not a cost per tonne.

Fixed costs cannot be written with continuous variables alone. Introduce one binary per unit,

    y_u = 1 if unit u is built, 0 otherwise

and the standard **fixed-charge** pair of a linear cost term and a big-M style capacity link:

    minimize    sum_u k_u F_u  +  sum_u f_u y_u
    subject to  F_u <= Fmax_u y_u          for all u in U
                (the balances and bounds as before)
                y_u in {0, 1}

The constraint `F_u <= Fmax_u y_u` is the entire mechanism: if `y_u = 0` the throughput is forced to zero, and if `y_u = 1` the ordinary capacity limit applies. The model is now a mixed-integer linear program. Its LP relaxation, obtained by replacing `y_u in {0,1}` by `0 <= y_u <= 1`, allows a unit to be "one third built" and pays only one third of its capital charge, which is why the relaxation is optimistic and why the gap between the two values matters. Week 5 studies exactly this: how the strength of a formulation, big-M against the convex hull, determines how large that gap is, and Week 11 studies how branch and bound closes it.

Here the capital charges are stated as an equivalent daily cost, that is annualized capital divided by operating days per year.


In [ ]:
# --- Unit selection: the MILP, and its LP relaxation -------------------------
FIXED = {"SMR": 22000.0, "ATR": 15000.0, "ELY": 9000.0,
         "PSA": 12000.0, "MEM": 30000.0}          # equivalent daily capital charge, USD/day
print("equivalent daily capital charge, USD/day")
print(pd.Series(FIXED).to_string())

milp = pick_solver("milp")
m_ip = build_super(binaries=True, name="superstructure_MILP")
r_ip = milp.solve(m_ip)
print("\n", r_ip.solver.termination_condition)
assert r_ip.solver.termination_condition == pyo.TerminationCondition.optimal, \
    "superstructure MILP did not solve to optimality"

F_ip = {u: pyo.value(m_ip.F[u]) for u in UNITS}
y_ip = {u: int(round(pyo.value(m_ip.y[u]))) for u in UNITS}
cost_ip = float(pyo.value(m_ip.cost))

m_rx = build_super(binaries=True, relax=True, name="superstructure_MILP_relaxation")
r_rx = lp.solve(m_rx)
assert r_rx.solver.termination_condition == pyo.TerminationCondition.optimal
cost_rx = float(pyo.value(m_rx.cost))

print(f"\nMILP optimum          : {cost_ip:>12,.2f} USD/day")
print(f"LP relaxation of it   : {cost_rx:>12,.2f} USD/day")
print(f"integrality gap       : {cost_ip - cost_rx:>12,.2f} USD/day "
      f"({100 * (cost_ip - cost_rx) / cost_ip:.2f} percent of the MILP value)")
assert cost_rx <= cost_ip + 1e-6, "the relaxation must be a lower bound"

print("\nfractional 'partly built' units in the relaxation:",
      {u: round(pyo.value(m_rx.y[u]), 4) for u in UNITS
       if 1e-6 < pyo.value(m_rx.y[u]) < 1 - 1e-6})

sel = pd.DataFrame({"LP throughput": F_lp, "MILP throughput": F_ip,
                    "MILP built": y_ip, "capital USD/day": FIXED})
print("\n", sel.round(4).to_string())

# Verify the MILP objective by recomputing it from the reported solution.
recomputed = sum(OPEX[u] * F_ip[u] for u in UNITS) + sum(FIXED[u] * y_ip[u] for u in UNITS)
print(f"\nobjective recomputed from the reported solution: {recomputed:,.2f} USD/day")
assert abs(recomputed - cost_ip) < 1e-4, "reported objective does not match the solution"
assert all(F_ip[u] <= CAP[u] * y_ip[u] + 1e-6 for u in UNITS), "a fixed-charge link is violated"

### 7.2 Load-dependent recovery: the superstructure becomes an MINLP

The linear model used one recovery figure per purifier, the value at design throughput. That is an approximation. A real adsorption or membrane separation recovers a larger *fraction* of the hydrogen when it is run below design, because contact time per tonne is longer. A first-order representation is a recovery that falls linearly with load,

    eta_u(F_u) = eta0_u - s_u F_u

so that the purified hydrogen leaving unit `u` is

    eta_u(F_u) F_u = eta0_u F_u - s_u F_u^2                              [t/day]

The coefficients below are set so that `eta_u(Fmax_u)` reproduces exactly the design recovery used in Sections 7 and 7.1, which makes the three models directly comparable: `0.92 - 0.0004 * 100 = 0.88` for `PSA` and `0.84 - 0.0005 * 80 = 0.80` for `MEM`.

The product balance is now a quadratic equality. That single term changes the class of the problem completely.

- With the binaries removed it is an **NLP**, and because the equality is quadratic the feasible set is not convex, so a local solver returns a local solution with no global guarantee.
- With the binaries present it is an **MINLP**, and a nonconvex one. This is the hardest class treated in this course.

The full model is

    minimize    sum_u k_u F_u + sum_u f_u y_u
    subject to  F_SMR + F_ATR = F_PSA + F_MEM
                sum_{u in {PSA, MEM}} (eta0_u - s_u F_u) F_u + F_ELY = D
                F_u <= Fmax_u y_u,   0 <= F_u <= Fmax_u,   y_u in {0, 1}

This is the canonical shape of a process synthesis problem, and it is why superstructure optimization is an MINLP subject rather than an LP subject. Week 5 supplies the discrete modeling that produces the `y` variables and the formulation-strength question, and Week 11 supplies the branch and bound machinery that solves them.

The cell below solves the MINLP with a dedicated solver and then, because an MINLP answer should never be taken on trust, cross-checks it by enumerating all `2^5 = 32` unit-selection patterns and solving the resulting continuous NLP for each one. Complete enumeration is exactly what branch and bound is designed to avoid, but with five binaries it is cheap and it settles the question.


In [ ]:
# --- Load-dependent recovery: solve the MINLP, then verify by enumeration ----
import itertools

ETA0      = {"PSA": 0.92,   "MEM": 0.84}     # recovery extrapolated to zero load
ETA_SLOPE = {"PSA": 0.0004, "MEM": 0.0005}   # loss of recovery per t/day of load

for u in ("PSA", "MEM"):
    check = ETA0[u] - ETA_SLOPE[u] * CAP[u]
    print(f"{u}: eta({CAP[u]:.0f} t/day) = {check:.4f}, design value {ETA_DES[u]:.4f}")
    assert abs(check - ETA_DES[u]) < 1e-12, "the load-dependent law must match at design load"

minlp = pick_solver("minlp")
m_nl = build_super(binaries=True, nonlinear=True, name="superstructure_MINLP")
r_nl = minlp.solve(m_nl)
print("\n", r_nl.solver.termination_condition)
assert r_nl.solver.termination_condition == pyo.TerminationCondition.optimal, \
    "superstructure MINLP did not solve to optimality"
F_nl = {u: max(pyo.value(m_nl.F[u]), 0.0) for u in UNITS}
y_nl = {u: int(round(pyo.value(m_nl.y[u]))) for u in UNITS}
cost_nl = float(pyo.value(m_nl.cost))
print(f"MINLP optimum: {cost_nl:,.2f} USD/day, units built "
      f"{[u for u in UNITS if y_nl[u]]}")

# --- Independent check: enumerate all 32 selections, solve each NLP with Ipopt
nlp = pick_solver("nlp")
best_cost, best_pattern, best_F = np.inf, None, None
n_feasible = 0
for bits in itertools.product([0, 1], repeat=len(UNITS)):
    fy = dict(zip(UNITS, bits))
    mm = build_super(binaries=True, nonlinear=True, fix_y=fy, name="enum")
    try:
        rr = nlp.solve(mm, load_solutions=False)     # do not load a failed solve
    except Exception:
        continue
    if rr.solver.termination_condition != pyo.TerminationCondition.optimal:
        continue
    mm.solutions.load_from(rr)
    Fv = {u: pyo.value(mm.F[u]) for u in UNITS}
    if any(Fv[u] > CAP[u] * fy[u] + 1e-6 for u in UNITS) or any(Fv[u] < -1e-6 for u in UNITS):
        continue
    pure = (sum((ETA0[u] - ETA_SLOPE[u] * Fv[u]) * Fv[u] for u in ("PSA", "MEM"))
            + Fv["ELY"])
    if abs(pure - DEMAND) > 1e-5:
        continue
    n_feasible += 1
    v = float(pyo.value(mm.cost))
    if v < best_cost:
        best_cost, best_pattern, best_F = v, fy, Fv

print(f"\nenumeration: {n_feasible} of 32 unit-selection patterns are feasible")
print(f"best by enumeration : {best_cost:,.2f} USD/day, units built "
      f"{[u for u in UNITS if best_pattern[u]]}")
print(f"MINLP solver        : {cost_nl:,.2f} USD/day")
rel = abs(best_cost - cost_nl) / abs(cost_nl)      # solver tolerances differ slightly
print(f"relative difference : {rel:.2e}")
assert rel < 1e-6, "MINLP solver and enumeration disagree on the optimal value"
assert best_pattern == y_nl, "MINLP solver and enumeration chose different units"
print("The MINLP solver and complete enumeration agree, on value and on structure.")

compare = pd.DataFrame({"LP (no capital charge)": F_lp,
                        "MILP (fixed recovery)": F_ip,
                        "MINLP (load-dependent recovery)": F_nl}).round(4)
print("\nunit throughput, t/day, under the three models")
print(compare.to_string())
print(f"\nMILP cost  {cost_ip:,.2f} USD/day, MINLP cost {cost_nl:,.2f} USD/day, "
      f"difference {cost_ip - cost_nl:,.2f} USD/day")

In [ ]:
# --- Figure 3: the superstructure and what each model selects ----------------
UPOS = {"SMR": (0.0, 2.25), "ATR": (0.0, 1.25), "ELY": (0.0, 0.25),
        "PSA": (1.5, 1.95), "MEM": (1.5, 0.95), "PROD": (3.0, 1.25)}
SUPER_ARCS = [("SMR", "PSA"), ("SMR", "MEM"), ("ATR", "PSA"), ("ATR", "MEM"),
              ("PSA", "PROD"), ("MEM", "PROD"), ("ELY", "PROD")]

fig, axes = plt.subplots(1, 2, figsize=(10.4, 3.9), constrained_layout=True,
                         gridspec_kw={"width_ratios": [1.25, 1.0]})

ax = axes[0]
active = {u: F_lp[u] > 1e-9 for u in UNITS}
for (i, j) in SUPER_ARCS:
    live = active.get(i, True) and (j == "PROD" or active.get(j, True))
    x0, y0 = UPOS[i]; x1, y1 = UPOS[j]
    col = PALETTE[0] if live else MIST
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0), zorder=2 if live else 1,
                arrowprops=dict(arrowstyle="-|>", color=col, lw=2.0 if live else 1.0,
                                shrinkA=24, shrinkB=24))
for u, (x, y) in UPOS.items():
    if u == "PROD":
        fc, lab = PALETTE[1], "PRODUCT\n100 t/day"
    else:
        fc = PALETTE[0] if active[u] else MIST
        lab = f"{u}\n{F_lp[u]:.0f} t/d"
    ax.add_patch(plt.Rectangle((x - 0.34, y - 0.20), 0.68, 0.40, facecolor=fc,
                               edgecolor=GRAPHITE, lw=1.0, zorder=4))
    ax.text(x, y, lab, ha="center", va="center", fontsize=7,
            color="white" if fc != MIST else INK, zorder=5)
ax.text(0.0, 2.85, "production", fontsize=8, color=GRAPHITE, ha="center")
ax.text(1.5, 2.85, "purification", fontsize=8, color=GRAPHITE, ha="center")
ax.text(3.0, 2.85, "product", fontsize=8, color=GRAPHITE, ha="center")
ax.text(-0.45, -0.35, "grey = route receives zero flow in the LP optimum",
        fontsize=7.5, color=GRAPHITE, ha="left")
ax.set_xlim(-0.5, 3.5); ax.set_ylim(-0.45, 3.1); ax.axis("off")
ax.set_title(f"Superstructure, LP optimum {cost_lp:,.0f} USD/day", fontsize=10, color=INK)

ax = tidy(axes[1])
idx = np.arange(len(UNITS)); w = 0.27
for k, (lab, series, col) in enumerate([("LP", F_lp, PALETTE[0]),
                                        ("MILP", F_ip, PALETTE[1]),
                                        ("MINLP", F_nl, PALETTE[2])]):
    ax.bar(idx + (k - 1) * w, [series[u] for u in UNITS], width=w, color=col, label=lab)
ax.plot(idx, [CAP[u] for u in UNITS], ls="none", marker="_", ms=16,
        color=GRAPHITE, label="capacity")
ax.set_xticks(idx); ax.set_xticklabels(UNITS)
ax.set_ylabel("throughput, t/day")
ax.set_title("Unit selection changes with the model class", fontsize=10)
ax.legend(fontsize=8, ncol=2)

fig.suptitle("Week 3, Figure 3: a hydrogen supply superstructure", color=INK, fontsize=10)
plt.show()

### 7.3 Interpretation of the superstructure results

**The linear superstructure selects routes without any binary variable.** The LP costs 177,250.00 USD/day, which is 1,772.50 USD per tonne of purified hydrogen. It runs `SMR` at its full 90 t/day, sends all of that plus 10 t/day of `ATR` crude through `PSA` at its full 100 t/day, sends the remaining 15 t/day of `ATR` crude through `MEM`, and leaves `ELY` at zero. Electrolysis is not chosen because it costs 3,400.00 USD per tonne of pure hydrogen against 1,704.55 for the `SMR` to `PSA` route. The optimizer reached that conclusion from the balances and the capacities alone. The dual of the product balance, 2,062.50 USD/t, is exactly the cost of the `ATR` to `MEM` route, which is the most expensive route actually carrying flow: the marginal tonne of product is made by the marginal route, which is the network reading of a shadow price from Week 2 transferred to a flowsheet.

**Capital charges change the flowsheet, not just the cost.** The MILP optimum is 248,720.00 USD/day and it builds `SMR`, `PSA` and `ELY` only. Both `ATR` and the membrane unit are dropped, and electrolysis, which the LP refused to use at all, is switched on to make up the 20.8 t/day that `SMR` and `PSA` cannot cover. That is the point of process synthesis: a route with a high running cost can still be correct if it avoids a large capital commitment. The LP relaxation values that plan at 222,232.14 USD/day, a gap of 26,487.86 USD/day or 10.65 percent, because a relaxed unit can be built fractionally and pay a fraction of its capital charge. Week 5 shows how a stronger formulation of the same logic shrinks that gap, and Week 11 shows what branch and bound does with it.

**One physically motivated nonlinearity turns the problem into an MINLP.** Making the purifier recovery depend on load replaces a linear equality with a quadratic one and moves the problem from MILP to nonconvex MINLP. The MINLP optimum is 247,496.00 USD/day, marginally below the MILP value, because `PSA` running at 90 t/day rather than its design 100 t/day recovers 0.884 instead of 0.880 and therefore needs slightly less expensive electrolysis behind it. The structural answer is unchanged here, which is a comfortable outcome and not a general one: in a nonconvex MINLP the selected structure can and does change with the nonlinearity, and there is no way to know without solving.

**Verification is not optional in this class of problem.** A nonconvex MINLP solver can return a local solution and report it as optimal. The cell above therefore does not rely on the solver alone: it enumerates all 32 unit-selection patterns, solves the continuous NLP behind each one, and confirms the same value and the same structure. Complete enumeration is affordable only because there are five binaries. Week 11 explains how branch and bound gets the same guarantee at a fraction of the work, and Week 6 makes the general point that any model whose provenance you cannot vouch for, including one you wrote yourself at midnight, has to be checked against an independent computation.


## 8. Exercises

**Exercise 1 (introductory).** Plant `R2` is shut down for a week. Set its supply to zero in the
transportation model, note that supply no longer matches demand, and repair the model by adding a dummy plant
whose shipping cost represents an emergency purchase at 55 USD/t. Re-solve, report the new cost, and identify
which gigafactory is served by the dummy.

**Exercise 2 (introductory).** In the transshipment network, raise the capacity of `R1 -> H1` from 400 to
500 t and re-solve. Report the change in cost and confirm that it equals the LP dual of that arc's upper
bound in the base solution, multiplied by the change, as long as the basis does not change.

**Exercise 3 (intermediate).** Solve the shortest path from `R1` to every gigafactory by changing only the
`b` vector, one solve per destination, and tabulate the results. Then obtain all four distances in a single
LP by sending 3 units out of `R1` with `b_Gj = -1` at three chosen destinations, and explain why the two
approaches do not in general give the same arc flows even though the total cost matches.

**Exercise 4 (intermediate).** Verify the max-flow min-cut theorem on a modified instance. Reduce the
capacity of `R1 -> H1` to 150 t, keep the halved terminal outbound capacities, re-solve the maximum flow
problem, and find the minimum cut by enumeration. Report the new maximum flow, the new cut, and the list of
arcs whose capacity is now worth increasing.

**Exercise 5 (advanced).** Break total unimodularity deliberately and measure the cost. Add a binary
`y_{ij}` to each terminal outbound arc with a fixed handling charge of 900 USD per arc used, and the
constraint `f_ij <= u_ij y_ij`. Solve the MILP and its LP relaxation. Report the integrality gap, the number
of branch and bound nodes reported by the solver, and compare the MILP solution with the pure LP solution of
section 2.2. Explain which structural property was lost and why the LP relaxation is now only a bound.

**Exercise 6 (introductory, superstructure).** In the linear superstructure of Section 7, the electrolyzer is never used. Find the operating cost of `ELY`, in USD/t, at which it just enters the optimal LP solution. Do this two ways: by a bisection search over at most 20 solves, and by comparing route costs by hand from the table printed in Section 7. The two answers must agree.

**Exercise 7 (intermediate, superstructure).** Add a carbon price of 90 USD per tonne of CO2 to the superstructure, with emission factors of 9.0 t CO2 per tonne of crude hydrogen from `SMR`, 7.5 from `ATR` and 0.4 from `ELY` (the last from grid electricity). Re-solve the LP, the MILP and the MINLP. Report the flowsheet selected in each case and the carbon price at which the MILP first stops building `SMR`.

**Exercise 8 (advanced, superstructure).** Replace the fixed capital charge `f_u y_u` by the six-tenths rule, `f_u (F_u / Fmax_u)^0.6 y_u`, which is the standard scaling law for process equipment cost. Explain why this is nonconvex, why a naive implementation is badly behaved at `F_u = 0`, and how the `y_u` variable together with a lower bound `F_u >= Fmin_u y_u` repairs the numerical difficulty. Solve the resulting MINLP with a global solver, cross-check against the 32-pattern enumeration used in Section 7.2, and report whether the selected flowsheet changes.

**Exercise 9 (introductory, cross-tool).** Build the transportation block of
`excel/W03_transportation_STUDENT.xlsx` as laid out above: cost matrix `C6:F8`, changing cells
`C12:F14`, supply `VALUE` column `G12:G14`, demand `VALUE` row `C15:F15`, objective cell `C19`. Solve
with OpenSolver and request the sensitivity report. Confirm that cell `C19` equals the minimum shipping
cost printed in Section 2.1 to the cent, that the shipment matrix agrees cell by cell with `m.x`, and
that the `Shadow Price` entries equal the supply and demand duals. Then attempt the shortest path from
`R1` to `G4` in the same workbook, stop as soon as you have identified the first thing the layout cannot
represent, and report in two sentences what that thing was.


## 9. Takeaways

- One template covers the whole family. Choose the graph, the costs, the capacities and the supply vector `b`, and transportation, transshipment, shortest path and maximum flow all follow from `min c'f` subject to `E f = b`, `0 <= f <= u`.
- The node balance is the model. Everything a network model can express passes through outflow minus inflow equals net supply, and the node-arc incidence matrix `E` records that structure with one `+1` and one `-1` per column.
- Network LPs return integers for free. Total unimodularity of `E` plus integral data gives integral basic solutions, which is why shortest path and assignment problems are solved as linear programs and not as integer programs.
- Total unimodularity is a property of the matrix, and it is fragile. One side constraint that does not belong to the incidence structure, or one fixed-charge binary, and fractional vertices reappear.
- Duals carry the engineering content. Node potentials price material at each location, arc reduced costs say which lanes should be empty or saturated, and in a maximum flow problem the non-zero capacity duals identify the minimum cut, which is the only place where extra capacity is worth anything.
- A process superstructure is a network flow model whose nodes are units and whose arcs are streams. Alternative process routes are embedded side by side in one model, and the choice between them is made by continuous split variables that the optimizer drives to zero, with no binary needed and no split fraction written as a ratio.
- Unit selection, not route selection, is what forces integer variables. The fixed-charge pair `F_u <= Fmax_u y_u` with a capital term `f_u y_u` makes the superstructure an MILP, and its LP relaxation is optimistic because a fractional `y_u` buys a fraction of a unit. Week 5 treats formulation strength and Week 11 treats branch and bound.
- One physically motivated nonlinearity, such as a recovery that depends on load, makes the same superstructure a nonconvex MINLP. That is the standard shape of a process synthesis problem, and it is why a local solve must always be cross-checked, here against complete enumeration of the 32 unit-selection patterns.
